In [18]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# Load your .env file
load_dotenv()

# Configuration Constants
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = "https://openrouter.ai/api" # The "Secret Sauce" for the Anthropic SDK
MODEL_NAME = "google/gemini-2.0-flash-001" # Fast and cheap for development

# Initialize the client once
client = Anthropic(
    base_url=BASE_URL,
    api_key=OPENROUTER_API_KEY
)

In [19]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": MODEL_NAME,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [20]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [21]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that takes an AWS region string (e.g., 'us-west-2') as input and returns a dictionary containing the default endpoint URLs for S3, EC2, and Lambda in that region. If the region is invalid, return an empty dictionary.", 'type': 'python'}, {'task': 'Generate a JSON object representing an IAM policy that grants read-only access to all S3 buckets in an AWS account.  Specify the necessary resources to grant that read only access.', 'type': 'json'}, {'task': 'Write a regular expression to validate if a string matches the format of an Amazon Resource Name (ARN) for an EC2 instance. Assume that you only need to perform basic ARN validation (i.e., arn:partition:service:region:account-id:resource-id) and are not checking service-specific ARN parts. This check should focus on correctly delimiting and identifying key parts of an ARN.', 'type': 'regex'}]


In [22]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [23]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [24]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [25]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [26]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [27]:
print(json.dumps(results, indent=2))


[
  {
    "output": "```python\nimport boto3\nfrom botocore.exceptions import ClientError, ParamValidationError\n\n\ndef get_default_endpoints(region):\n    \"\"\"\n    Returns a dictionary containing the default endpoint URLs for S3, EC2, and Lambda in the specified AWS region.\n\n    Args:\n        region (str): The AWS region string (e.g., 'us-west-2').\n\n    Returns:\n        dict: A dictionary with keys 's3', 'ec2', and 'lambda', each holding the corresponding endpoint URL for the given region.\n              Returns an empty dictionary if the region is invalid.\n    \"\"\"\n    endpoints = {}\n    try:\n        # S3\n        s3_client = boto3.client('s3', region_name=region)\n        endpoints['s3'] = s3_client.meta.endpoint_url\n\n        # EC2\n        ec2_client = boto3.client('ec2', region_name=region)\n        endpoints['ec2'] = ec2_client.meta.endpoint_url\n\n        # Lambda\n        lambda_client = boto3.client('lambda', region_name=region)\n        endpoints['lambda'] =